In [1]:
%%capture
!pip install -U lightautoml

In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [3]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [4]:
import torch
from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
from lightautoml.tasks import Task

In [5]:
N_THREADS = 4
N_FOLDS = 8
RANDOM_STATE = 42
TEST_SIZE = 0.2
TIMEOUT = 3600*100

numpy.random.seed(RANDOM_STATE)
torch.set_num_threads(N_THREADS)

train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv')
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [6]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [7]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].mean(), inplace=True)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].mean(), inplace=True)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [8]:
train = train.drop(columns=['id'])

In [9]:
task = Task('reg')

In [10]:
model = TabularAutoML(
    task = task,
    timeout = TIMEOUT,
    cpu_limit = N_THREADS,
    general_params = {'use_algos':[['cb', 'linear_l2', 'gbm', 'lgb', 'lgb_tuned']]},
    selection_params={'mode' : 0},
    tuning_params = {'max_tuning_time': 3600},
    reader_params = {'n_jobs': N_THREADS, 'cv': N_FOLDS, 'random_state': RANDOM_STATE}
)

out_of_fold_predictions = model.fit_predict(
    train,
    roles = {
        'target': 'efficiency',
        #'drop': ['unique_id']
        #'weights': 'weight'
    }, 
    verbose = 2
)

[06:52:39] Stdout logging level is INFO2.
[06:52:39] Task: reg

[06:52:39] Start automl preset with listed constraints:
[06:52:39] - time: 360000.00 seconds
[06:52:39] - CPU: 4 cores
[06:52:39] - memory: 16 GB

[06:52:39] Train data shape: (20000, 16)

[06:52:49] Layer 1 train process start. Time left 359989.81 secs
[06:52:49] Start fitting Lvl_0_Pipe_0_Mod_0_LinearL2 ...
[06:52:49] ===== Start working with fold 0 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[06:52:49] ===== Start working with fold 1 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[06:52:50] ===== Start working with fold 2 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[06:52:50] ===== Start working with fold 3 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[06:52:50] ===== Start working with fold 4 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[06:52:50] ===== Start working with fold 5 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[06:52:51] ===== Start working with fold 6 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[06:52:51] ===== Start working with fold 7 for Lvl_0_Pipe_

Optimization Progress: 100%|██████████| 101/101 [04:05<00:00,  2.43s/it, best_trial=57, best_value=-0.0114]

[06:57:09] Hyperparameters optimization for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM completed
[06:57:09] The set of hyperparameters {'feature_fraction': 0.8186372617753938, 'num_leaves': 25, 'bagging_fraction': 0.7043087697305205, 'min_sum_hessian_in_leaf': 0.006835363023862769, 'reg_alpha': 0.000267800604206848, 'reg_lambda': 3.596699639618822}
 achieve -0.0114 mse
[06:57:09] Start fitting Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM ...
[06:57:09] ===== Start working with fold 0 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====


[06:57:09] ===== Start working with fold 1 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[06:57:10] ===== Start working with fold 2 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[06:57:11] ===== Start working with fold 3 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[06:57:12] ===== Start working with fold 4 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[06:57:12] ===== Start working with fold 5 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[06:57:13] ===== Start working with fold 6 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[06:57:14] ===== Start working with fold 7 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[06:57:14] Fitting Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM finished. score = -0.010675617181116637
[06:57:14] Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM fitting and predicting completed
[06:57:14] Start fitting Lvl_0_Pipe_1_Mod_2_CatBoost ...
[06:57:14] ===== Start working with fold 0 for Lvl_0_Pipe_1_Mod_2_CatBoost =====
[06:57:15] ===== Start working with fold 1 for Lvl_0_Pipe_1_Mod_2_CatBoost =====
[

In [11]:
#print(f'Concordance index (lifelines): {100*(1-numpy.sqrt(mean_squared_error(y_test, model.predict(X_test))))}')

In [12]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = model.predict(test)
test_predictions

array([[0.40710175],
       [0.5413269 ],
       [0.52207994],
       ...,
       [0.6152621 ],
       [0.445878  ],
       [0.5594516 ]], dtype=float32)

In [13]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions.data.reshape(-1),
})
submission

,id,efficiency
0,0,0.407102
1,1,0.541327
2,2,0.522080
3,3,0.469803
4,4,0.457785
...,...,...
11995,11995,0.553457
11996,11996,0.466045
11997,11997,0.615262
11998,11998,0.445878


In [14]:
submission.to_csv('submission.csv', index = False)